# 11 — Missing Data

Missing data is in every real-world dataset. This notebook covers detection, removal,
imputation, interpolation, and the critical distinction between NaN, None, NaT, and pd.NA.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 300

# Employee dataset with intentional missing values
employees = pd.DataFrame({
    'emp_id':     range(1001, 1001 + n),
    'name':       [f'Employee_{i:03d}' for i in range(n)],
    'department': np.random.choice(['Engineering', 'Sales', 'HR', 'Marketing', None], n,
                                   p=[0.3, 0.25, 0.15, 0.2, 0.1]),
    'salary':     np.where(
                      np.random.random(n) > 0.08,
                      np.random.randint(40000, 150000, n).astype(float),
                      np.nan
                  ),
    'experience_years': np.where(
                            np.random.random(n) > 0.05,
                            np.random.randint(1, 20, n).astype(float),
                            np.nan
                        ),
    'join_date':  pd.to_datetime([
                      d if np.random.random() > 0.05 else None
                      for d in pd.date_range('2010-01-01', periods=n, freq='W')
                  ]),
    'rating':     np.random.choice([1.0, 2.0, 3.0, 4.0, 5.0, np.nan], n,
                                   p=[0.05, 0.1, 0.25, 0.35, 0.15, 0.10]),
    'city':       np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', None], n,
                                   p=[0.25, 0.25, 0.20, 0.20, 0.10]),
})

print(employees.info())

## 1. Types of Missing Data in Pandas

| Sentinel | Type | Represents |
|----------|------|------------|
| `np.nan` | float | Classic NaN — contagious in math |
| `None` | Python object | Python null — in object columns |
| `pd.NaT` | datetime | Not a Time |
| `pd.NA` | NAType | New universal NA (Pandas 1.0+) |

In [ ]:
# NaN is float — contagious in arithmetic
print(f"np.nan + 5 = {np.nan + 5}")
print(f"np.nan == np.nan: {np.nan == np.nan}")  # False! Use isna()

# pd.NaT for datetime
dt = pd.to_datetime(['2023-01-01', None])
print(f"\nNaT in datetime: {dt}")

# pd.NA — universal missing marker (nullable types)
s = pd.Series([1, 2, None], dtype='Int64')
print(f"\npd.NA: {s[2]}")
print(f"pd.NA + 1 = {s[2] + 1}")  # NA (propagates)

## 2. Detecting Missing Values

In [ ]:
# Per-column missing count
missing_count = employees.isna().sum()
missing_pct   = (employees.isna().mean() * 100).round(1)

missing_report = pd.DataFrame({'count': missing_count, 'pct': missing_pct})
print(missing_report[missing_report['count'] > 0])

In [ ]:
# Rows with ANY missing value
any_missing = employees[employees.isna().any(axis=1)]
print(f"Rows with any NaN: {len(any_missing)} / {len(employees)}")

# Rows where ALL values are missing (truly empty rows)
all_missing = employees[employees.isna().all(axis=1)]
print(f"Rows with ALL NaN: {len(all_missing)}")

In [ ]:
# notna() — inverse of isna()
print(f"Rows with valid salary: {employees['salary'].notna().sum()}")

# Count non-null per column (also from info())
print(employees.count())  # excludes NaN

## 3. dropna() — Remove Missing Rows/Columns

In [ ]:
# Drop rows with ANY NaN (aggressive)
no_nan = employees.dropna()
print(f"Original: {len(employees)}  After dropna(how='any'): {len(no_nan)}")

# Drop rows where ALL values are NaN (conservative)
no_all_nan = employees.dropna(how='all')
print(f"After dropna(how='all'): {len(no_all_nan)}")

In [ ]:
# subset: only consider specific columns for NA check
clean_salary = employees.dropna(subset=['salary', 'department'])
print(f"After dropna(subset=['salary','department']): {len(clean_salary)}")

# thresh: keep rows with at least N non-NaN values
min_5_valid = employees.dropna(thresh=5)
print(f"After dropna(thresh=5): {len(min_5_valid)}")

In [ ]:
# axis=1: drop columns with any NaN
no_nan_cols = employees.dropna(axis=1)
print(f"Columns remaining after dropna(axis=1): {no_nan_cols.columns.tolist()}")

## 4. fillna() — Imputing Missing Values

In [ ]:
# Scalar fill
df_filled = employees.copy()
df_filled['city'] = df_filled['city'].fillna('Unknown')
df_filled['department'] = df_filled['department'].fillna('Not Specified')
print(df_filled['city'].value_counts())

In [ ]:
# Statistical fill — fill with mean/median (numeric columns)
df_filled['salary'] = df_filled['salary'].fillna(df_filled['salary'].median())
df_filled['rating'] = df_filled['rating'].fillna(df_filled['rating'].mean().round(1))
print(f"Salary NaN after fill: {df_filled['salary'].isna().sum()}")
print(f"Rating NaN after fill: {df_filled['rating'].isna().sum()}")

In [ ]:
# Group-based fill — fill with department median salary
df_temp = employees.copy()
df_temp['salary'] = df_temp.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median())
)
# Remaining NaN (dept was also NaN): fill with global median
df_temp['salary'] = df_temp['salary'].fillna(df_temp['salary'].median())
print(f"Salary NaN after group fill: {df_temp['salary'].isna().sum()}")

In [ ]:
# Forward fill and backward fill (ffill / bfill)
# Classic use case: time series data
ts = pd.Series([10, np.nan, np.nan, 40, np.nan, 60],
               index=pd.date_range('2023-01', periods=6, freq='ME'))
print("Original:")
print(ts.to_frame('value'))

print("\nffill (propagate last valid):")
print(ts.ffill().to_frame('value'))

print("\nbfill (propagate next valid):")
print(ts.bfill().to_frame('value'))

In [ ]:
# limit: max consecutive fills
print("ffill(limit=1) — fill max 1 consecutive NaN:")
print(ts.ffill(limit=1).to_frame('value'))

## 5. interpolate() — Smooth Imputation

In [ ]:
ts2 = pd.Series([10.0, np.nan, np.nan, 40.0, np.nan, 60.0])

print("linear interpolation:")
print(ts2.interpolate(method='linear'))

print("\npolynomial (order=2):")
print(ts2.interpolate(method='polynomial', order=2))

print("\nnearest:")
print(ts2.interpolate(method='nearest'))

In [ ]:
# Real-world: stock price interpolation
stock_price = pd.Series(
    [142.5, np.nan, np.nan, 145.2, np.nan, 148.0, np.nan, 150.5],
    index=pd.date_range('2023-01-02', periods=8, freq='B'),
    name='price'
)

print("Stock price with interpolation:")
result = pd.DataFrame({
    'original':     stock_price,
    'linear_interp': stock_price.interpolate('linear'),
    'ffill':        stock_price.ffill()
})
print(result)

## 6. fillna() with a Dict — Different Fill per Column

In [ ]:
fill_values = {
    'department': 'Unknown',
    'salary':     employees['salary'].median(),
    'rating':     employees['rating'].mean().round(1),
    'city':       'Unknown',
}

df_filled_all = employees.fillna(fill_values)
print("Missing counts after fill:")
print(df_filled_all.isna().sum())

## 7. Missing Data Patterns — MCAR, MAR, MNAR

In [ ]:
# Analyze whether missing salary correlates with department (MAR pattern)
employees['salary_missing'] = employees['salary'].isna().astype(int)

missing_by_dept = employees.groupby('department', dropna=False)['salary_missing'].mean()
print("Missing salary rate by department:")
print(missing_by_dept.round(3))
print("\nIf rates differ significantly, data is MAR (Missing At Random — depends on department)")

employees.drop(columns=['salary_missing'], inplace=True)

## 8. isna() vs isnull() — and notna() vs notnull()

In [ ]:
# isna() and isnull() are identical aliases
s = pd.Series([1.0, np.nan, 3.0, None])

print(f"isna():   {s.isna().tolist()}")
print(f"isnull(): {s.isnull().tolist()}")
print(f"Identical: {(s.isna() == s.isnull()).all()}")

# notna() and notnull() are identical
print(f"\nnotna():   {s.notna().tolist()}")
print(f"notnull(): {s.notnull().tolist()}")

## 9. Real-World Missing Data Strategy

In [ ]:
def handle_missing_employees(df: pd.DataFrame) -> pd.DataFrame:
    """Production-quality missing data handler for employee dataset."""
    df = df.copy()

    # 1. Audit first
    missing = df.isna().sum()
    print("Missing before:")
    print(missing[missing > 0])

    # 2. Categorical: fill with mode or 'Unknown'
    for col in ['department', 'city']:
        df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'Unknown')

    # 3. Numeric: group-aware median fill
    df['salary'] = df.groupby('department', dropna=False)['salary'].transform(
        lambda x: x.fillna(x.median())
    ).fillna(df['salary'].median())

    df['experience_years'] = df['experience_years'].fillna(
        df['experience_years'].median()
    )

    # 4. Rating: fill with group mean
    df['rating'] = df.groupby('department', dropna=False)['rating'].transform(
        lambda x: x.fillna(x.mean())
    ).round(1)

    # 5. Datetime: forward fill within sorted order
    df = df.sort_values('emp_id')
    df['join_date'] = df['join_date'].ffill()

    print("\nMissing after:")
    print(df.isna().sum()[df.isna().sum() > 0])
    return df


clean_employees = handle_missing_employees(employees)

## 10. Quick Reference — Missing Data Methods

| Method | Use Case |
|--------|----------|
| `isna()` / `notna()` | Detect missing values |
| `dropna(how='any')` | Remove rows with any NaN |
| `dropna(subset=['col'])` | Remove rows with NaN in specific column |
| `dropna(thresh=N)` | Keep rows with at least N non-NaN |
| `fillna(value)` | Replace NaN with scalar |
| `fillna(dict)` | Per-column fill values |
| `ffill()` | Forward fill (prev valid value) |
| `bfill()` | Backward fill (next valid value) |
| `interpolate()` | Smooth numeric imputation |
| `groupby().transform(fillna)` | Group-aware imputation |